In [ ]:
import math
import warnings
from dataclasses import dataclass
from typing import Optional, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import akshare as ak
import yfinance as yf

# Set default font configuration
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print("✓ Libraries imported successfully")

In [ ]:
def macd_series(close, fast=12, slow=26, signal=9) -> pd.DataFrame:
    """Compute MACD indicator - enhanced version"""
    # Force conversion to Series
    if isinstance(close, pd.DataFrame):
        if len(close.columns) == 1:
            close = close.iloc[:, 0]
        else:
            close = close['Close']
    
    close = pd.Series(close).squeeze()
    
    # Ensure the series has an index
    if not hasattr(close, 'index') or close.index is None:
        close = pd.Series(close.values)
    
    # Compute MACD
    ema_fast = close.ewm(span=fast, adjust=False, min_periods=fast).mean()
    ema_slow = close.ewm(span=slow, adjust=False, min_periods=slow).mean()
    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=signal, adjust=False, min_periods=signal).mean()
    hist = macd - macd_signal
    
    # Build DataFrame with the original index
    return pd.DataFrame({
        "MACD": macd.values, 
        "MACD_Signal": macd_signal.values, 
        "MACD_Hist": hist.values
    }, index=close.index)

def bbands_series(close, window=20, sd=2.0) -> pd.DataFrame:
    """Compute Bollinger Bands - enhanced version"""
    # Force conversion to Series
    if isinstance(close, pd.DataFrame):
        if len(close.columns) == 1:
            close = close.iloc[:, 0]
        else:
            close = close['Close']
    
    close = pd.Series(close).squeeze()
    
    # Ensure the series has an index
    if not hasattr(close, 'index') or close.index is None:
        close = pd.Series(close.values)
    
    # Compute Bollinger Bands
    mid = close.rolling(window, min_periods=window).mean()
    std = close.rolling(window, min_periods=window).std(ddof=0)
    up = mid + sd * std
    lo = mid - sd * std
    
    # Build DataFrame with the original index
    return pd.DataFrame({
        "BB_Middle": mid.values, 
        "BB_Upper": up.values, 
        "BB_Lower": lo.values
    }, index=close.index)

print("✓ Technical indicator functions defined (enhanced version)")

In [ ]:
def signals_macd(df: pd.DataFrame, long_short=False) -> pd.Series:
    """
    MACD crossover strategy
    - Long-only mode: hold a long position when MACD > Signal, otherwise stay flat
    - Long-short mode: go long when MACD > Signal, go short when MACD < Signal
    """
    macd = df["MACD"]
    signal = df["MACD_Signal"]
    valid = macd.notna() & signal.notna()
    pos = np.where(macd > signal, 1, (-1 if long_short else 0))
    pos = np.where(np.isclose(macd, signal, equal_nan=False), 0, pos)
    pos = np.where(valid, pos, 0)
    return pd.Series(pos, index=df.index, name="position")

print("✓ MACD strategy function defined")

In [ ]:
def signals_bbands(df: pd.DataFrame, long_short=False) -> pd.Series:
    """
    Bollinger Bands breakout strategy
    - Long-only mode: go long on an upper-band breakout, exit when price falls below the middle band
    - Long-short mode: go long on an upper-band breakout, go short on a lower-band breakdown, exit on a middle-band reversion
    """
    close = df["Close"]
    up = df["BB_Upper"]
    lo = df["BB_Lower"]
    mid = df["BB_Middle"]
    
    pos = np.zeros(len(df), dtype=int)
    for i in range(1, len(df)):
        p = pos[i-1]
        if long_short:
            if close.iloc[i] > up.iloc[i]:
                p = 1
            elif close.iloc[i] < lo.iloc[i]:
                p = -1
            elif (close.iloc[i-1] >= mid.iloc[i-1]) and (close.iloc[i] < mid.iloc[i]):
                p = 0
            elif (close.iloc[i-1] <= mid.iloc[i-1]) and (close.iloc[i] > mid.iloc[i]):
                p = 0
        else:
            if close.iloc[i] > up.iloc[i]:
                p = 1
            elif close.iloc[i] < mid.iloc[i]:
                p = 0
        pos[i] = p
    return pd.Series(pos, index=df.index, name="position")

print("✓ Bollinger Bands strategy function defined")

In [ ]:
def signals_combo(df: pd.DataFrame, long_short=False) -> pd.Series:
    """
    Combined MACD and Bollinger Bands strategy
    - Long: price breaks above the upper band and MACD > Signal
    - Short: price breaks below the lower band and MACD < Signal (long-short mode only)
    - Exit: price reverts to the middle band or MACD reverses direction
    """
    close = df["Close"]
    up = df["BB_Upper"]
    lo = df["BB_Lower"]
    mid = df["BB_Middle"]
    macd = df["MACD"]
    sig = df["MACD_Signal"]
    
    pos = np.zeros(len(df), dtype=int)
    for i in range(1, len(df)):
        p = pos[i-1]
        # Exit conditions
        if p == 1 and ((close.iloc[i] < mid.iloc[i]) or (macd.iloc[i] < sig.iloc[i])):
            p = 0
        if p == -1 and ((close.iloc[i] > mid.iloc[i]) or (macd.iloc[i] > sig.iloc[i])):
            p = 0
        # Entry conditions
        if (close.iloc[i] > up.iloc[i]) and (macd.iloc[i] > sig.iloc[i]):
            p = 1
        elif long_short and (close.iloc[i] < lo.iloc[i]) and (macd.iloc[i] < sig.iloc[i]):
            p = -1
        pos[i] = p
    return pd.Series(pos, index=df.index, name="position")

print("✓ Combined strategy function defined")

In [ ]:
@dataclass
class Metrics:
    """Performance metrics data class"""
    cagr: float          # Compound annual growth rate
    ann_vol: float       # Annualized volatility
    sharpe: float        # Sharpe ratio
    maxdd: float         # Maximum drawdown
    win_rate: float      # Win rate
    n_trades: int        # Number of trades
    total_return: float  # Total return

print("✓ Metrics data class defined")

In [ ]:
def compute_trade_stats(position: pd.Series, market_ret: pd.Series, cost_bps: float) -> Tuple[int, float]:
    """Count trades and win rate using executed positions"""
    position = position.fillna(0.0).astype(float)
    market_ret = market_ret.fillna(0.0).astype(float)
    cost_rate = cost_bps / 10000.0
    trade_returns = []
    trade_equity = None

    for i in range(len(position)):
        curr_pos = float(position.iloc[i])
        prev_pos = float(position.iloc[i - 1]) if i > 0 else 0.0

        if prev_pos != 0.0 and curr_pos != prev_pos and trade_equity is not None:
            exit_cost = abs(prev_pos) * cost_rate
            trade_equity *= (1.0 - exit_cost)
            trade_returns.append(trade_equity - 1.0)
            trade_equity = None

        if curr_pos != 0.0 and curr_pos != prev_pos:
            entry_cost = abs(curr_pos) * cost_rate
            trade_equity = 1.0
            trade_equity *= (1.0 + curr_pos * market_ret.iloc[i] - entry_cost)
        elif curr_pos != 0.0 and trade_equity is not None:
            trade_equity *= (1.0 + curr_pos * market_ret.iloc[i])

    if trade_equity is not None:
        trade_returns.append(trade_equity - 1.0)

    n_trades = len(trade_returns)
    win_rate = float(np.mean(np.array(trade_returns) > 0)) if trade_returns else 0.0
    return n_trades, win_rate

def compute_metrics(
    equity: pd.Series,
    rets: pd.Series,
    position: pd.Series,
    market_ret: pd.Series,
    cost_bps: float,
) -> Metrics:
    """Compute strategy performance metrics"""
    af = 252  # Annualization factor (trading days)
    
    # Total return and CAGR
    total_ret = equity.iloc[-1] / equity.iloc[0] - 1.0
    yrs = max((equity.index[-1] - equity.index[0]).days / 365.25, 1e-9)
    cagr = (1 + total_ret) ** (1 / yrs) - 1 if yrs > 0 else 0
    
    # Volatility and Sharpe ratio
    ann_vol = rets.std(ddof=0) * math.sqrt(af)
    sharpe = (rets.mean() * af) / (ann_vol + 1e-12)
    
    # Maximum drawdown
    roll_max = equity.cummax()
    dd = equity / roll_max - 1.0
    maxdd = dd.min()
    
    # Trade statistics
    n_trades, win_rate = compute_trade_stats(position, market_ret, cost_bps)
    
    return Metrics(cagr, ann_vol, sharpe, maxdd, win_rate, n_trades, total_ret)

print("✓ Metrics calculation functions defined")

In [ ]:
def backtest(df: pd.DataFrame, cost_bps: float = 0.0, long_short: bool = False) -> Tuple[pd.DataFrame, Metrics]:
    """
    Run strategy backtest
    
    Parameters:
        df: DataFrame containing Close and position columns
        cost_bps: Transaction cost in basis points, e.g. 2 means 0.02%
        long_short: Whether the strategy supports long-short positions
    
    Returns:
        result: DataFrame with daily returns and equity curve
        metrics: Performance metrics object
    """
    px = df["Close"].astype(float)
    signal_pos = df["position"].astype(float)
    pos = signal_pos.shift(1).fillna(0.0)  # Executed position after a one-day signal delay
    ret = px.pct_change().fillna(0.0)
    
    # Calculate transaction costs from executed position changes
    turnover = pos.diff().abs().fillna(pos.abs())
    cost = turnover * (cost_bps / 10000.0)  # One-way cost
    
    # Strategy return = executed position * market return - transaction cost
    strat_ret = pos * ret - cost
    
    # Equity curve
    equity = (1.0 + strat_ret).cumprod()
    
    # Compute metrics
    metrics = compute_metrics(equity, strat_ret, pos, ret, cost_bps)
    
    # Build output DataFrame
    result = pd.DataFrame({
        "Close": px,
        "signal_position": signal_pos,
        "position": pos,
        "ret": ret,
        "cost": cost,
        "strat_ret": strat_ret,
        "equity": equity,
        "turnover": turnover
    }, index=df.index)
    
    return result, metrics

print("✓ Backtest engine defined")

In [ ]:
def load_cn_index(symbol_code: str, start_date: str) -> pd.DataFrame:
    """
    Load China index data via AkShare
    
    Parameters:
        symbol_code: Index code, e.g. "sh000001" (SSE Composite Index)
        start_date: Start date in "YYYY-MM-DD" format
    
    Returns:
        Standardized OHLCV DataFrame
    """
    print(f"  Loading {symbol_code}...")
    df_raw = ak.stock_zh_index_daily(symbol=symbol_code)
    
    # Standardize column names
    df = df_raw.rename(columns={
        'date': 'Date',
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close',
        'volume': 'Volume'
    })
    
    # Process dates
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.set_index('Date')
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
    
    # Convert data types
    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Filter time range
    df = df[df.index >= start_date]
    
    return df

print("✓ China index loader defined")

In [ ]:
def load_nikkei(start_date: str) -> pd.DataFrame:
    """
    Load Nikkei 225 index via Yahoo Finance
    
    Parameters:
        start_date: Start date in "YYYY-MM-DD" format
    
    Returns:
        Standardized OHLCV DataFrame
    """
    print(f"  Loading Nikkei 225...")
    df = yf.download("^N225", start=start_date, auto_adjust=False, progress=False)
    
    if df.empty:
        raise ValueError("Failed to load Nikkei 225 data")
    
    # Handle multi-level columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    # Select required columns
    df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
    
    # Ensure index is DatetimeIndex
    df.index = pd.to_datetime(df.index)
    
    return df

print("✓ Nikkei index loader defined")

In [ ]:
def load_dce_futures(symbol: str) -> pd.DataFrame:
    """
    Load Dalian Commodity Exchange continuous futures via AkShare
    
    Parameters:
        symbol: Contract code, e.g. "m0" (soybean meal), "c0" (corn), "j0" (coke), "i0" (iron ore)
    
    Returns:
        Standardized OHLCV DataFrame
    """
    print(f"  Loading DCE futures {symbol}...")
    df_raw = ak.futures_main_sina(symbol=symbol)
    
    # Map source column names to standardized labels
    colmap = {}
    for c in df_raw.columns:
        if "日期" in c: 
            colmap[c] = "Date"
        if "开盘" in c: 
            colmap[c] = "Open"
        if "最高" in c: 
            colmap[c] = "High"
        if "最低" in c: 
            colmap[c] = "Low"
        if "收盘" in c: 
            colmap[c] = "Close"
        if "成交" in c: 
            colmap[c] = "Volume"
    
    df = df_raw.rename(columns=colmap)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").set_index("Date")
    
    # Fill missing volume with 0 if needed
    if "Volume" not in df.columns:
        df["Volume"] = 0.0
    
    df = df[["Open", "High", "Low", "Close", "Volume"]]
    return df

print("✓ DCE futures loader defined")

In [ ]:
# Configuration
START_DATE = "2010-01-01"

# Define markets to test
MARKETS = {
    "SSE Index": ("cn_index", "sh000001"),
    "Nikkei 225": ("nikkei", None),
    "Soybean Meal": ("dce", "m0"),
    "Corn": ("dce", "c0"),
    "Coke": ("dce", "j0"),
    "Iron Ore": ("dce", "i0"),
}

# Start loading data
print("=" * 60)
print("Loading market data...")
print("=" * 60)

data_dict = {}

for market_name, (source, code) in MARKETS.items():
    try:
        if source == "cn_index":
            df = load_cn_index(code, START_DATE)
        elif source == "nikkei":
            df = load_nikkei(START_DATE)
        elif source == "dce":
            df = load_dce_futures(code)
            df = df[df.index >= START_DATE]
        
        # Ensure correct data types
        for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Remove NaN rows
        df = df.dropna(subset=['Close'])
        
        data_dict[market_name] = df
        print(f"  ✓ {market_name}: {len(df)} records")
        print(f"    Period: {df.index.min().date()} to {df.index.max().date()}")
        
    except Exception as e:
        print(f"  ✗ {market_name}: Loading failed - {str(e)}")

print("=" * 60)
print(f"Data loading completed, {len(data_dict)} markets loaded successfully")
print("=" * 60)

In [ ]:
def plot_market_analysis(df: pd.DataFrame, result: pd.DataFrame, title: str):
    """
    Plot market analysis with 4 subplots (No titles to avoid Chinese display issues)
    
    Parameters:
        df: DataFrame with indicators
        result: Backtest results
        title: Chart title (will be used in filename only, not displayed)
    """
    fig, axes = plt.subplots(4, 1, figsize=(16, 12))
    # Removed fig.suptitle to avoid Chinese character issues
    
    # ========== 1. Price & Bollinger Bands ==========
    ax1 = axes[0]
    ax1.plot(df.index, df['Close'], label='Close Price', linewidth=1.5, color='black')
    ax1.plot(df.index, df['BB_Upper'], '--', label='BB Upper', alpha=0.7, color='red', linewidth=1)
    ax1.plot(df.index, df['BB_Middle'], '--', label='BB Middle', alpha=0.7, color='blue', linewidth=1)
    ax1.plot(df.index, df['BB_Lower'], '--', label='BB Lower', alpha=0.7, color='red', linewidth=1)
    ax1.fill_between(df.index, df['BB_Upper'], df['BB_Lower'], alpha=0.1, color='gray')
    ax1.set_ylabel('Price', fontsize=11)
    # Removed ax1.set_title
    ax1.legend(loc='best', fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # ========== 2. MACD Indicator ==========
    ax2 = axes[1]
    ax2.plot(df.index, df['MACD'], label='MACD', linewidth=1.5, color='blue')
    ax2.plot(df.index, df['MACD_Signal'], label='Signal Line', linewidth=1.5, color='red')
    colors = ['green' if x > 0 else 'red' for x in df['MACD_Hist']]
    ax2.bar(df.index, df['MACD_Hist'], label='Histogram', alpha=0.4, color=colors, width=1)
    ax2.axhline(0, color='black', linewidth=0.8, linestyle='-', alpha=0.5)
    ax2.set_ylabel('MACD', fontsize=11)
    # Removed ax2.set_title
    ax2.legend(loc='best', fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    # ========== 3. Position Signals ==========
    ax3 = axes[2]
    ax3.plot(result.index, result['position'], linewidth=1.5, color='purple', drawstyle='steps-post')
    ax3.fill_between(result.index, 0, result['position'], alpha=0.3, color='purple', step='post')
    ax3.set_ylabel('Position', fontsize=11)
    # Removed ax3.set_title
    ax3.set_ylim([-1.5, 1.5])
    ax3.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax3.grid(True, alpha=0.3)
    # Add text annotation instead of title
    ax3.text(0.02, 0.95, '1=Long, -1=Short, 0=Flat', transform=ax3.transAxes, 
             fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    # ========== 4. Equity Curve ==========
    ax4 = axes[3]
    ax4.plot(result.index, result['equity'], linewidth=2, color='darkgreen', label='Strategy Equity')
    ax4.fill_between(result.index, 1, result['equity'], 
                     where=(result['equity'] >= 1), alpha=0.3, color='green', interpolate=True)
    ax4.fill_between(result.index, 1, result['equity'], 
                     where=(result['equity'] < 1), alpha=0.3, color='red', interpolate=True)
    ax4.axhline(1, color='black', linewidth=0.8, linestyle='--', alpha=0.5, label='Initial Capital')
    ax4.set_ylabel('Equity', fontsize=11)
    ax4.set_xlabel('Date', fontsize=11)
    # Removed ax4.set_title
    ax4.legend(loc='best', fontsize=9)
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print("✓ Visualization function defined (No titles)")

In [ ]:
print("\n" + "=" * 80)
print("Strategy 1: MACD Crossover Strategy")
print("=" * 80)

results_macd = {}
metrics_macd = {}

for market_name, df in data_dict.items():
    print(f"\n{'='*60}")
    print(f"Processing Market: {market_name}")
    print('='*60)
    
    # Determine if futures (long-short enabled)
    is_futures = market_name in ["Soybean Meal", "Corn", "Coke", "Iron Ore"]
    
    # Calculate technical indicators
    macd = macd_series(df["Close"])
    bb = bbands_series(df["Close"])
    feat = pd.concat([df, macd, bb], axis=1)
    
    # Generate trading signals
    pos = signals_macd(feat, long_short=is_futures)
    feat["position"] = pos
    
    # Run backtest
    result, metrics = backtest(feat, cost_bps=0.0, long_short=is_futures)
    
    # Save results
    results_macd[market_name] = result
    metrics_macd[market_name] = metrics
    
    # Print performance metrics
    print(f"\n[Performance Metrics]")
    print(f"  Total Return:      {metrics.total_return:>8.2%}")
    print(f"  CAGR:              {metrics.cagr:>8.2%}")
    print(f"  Sharpe Ratio:      {metrics.sharpe:>8.2f}")
    print(f"  Max Drawdown:      {metrics.maxdd:>8.2%}")
    print(f"  Win Rate:          {metrics.win_rate:>8.2%}")
    print(f"  Number of Trades:  {metrics.n_trades:>8d}")
    
    # Plot analysis
    plot_market_analysis(feat, result, f"{market_name} - MACD Strategy")

print("\n" + "=" * 80)
print("✓ MACD Strategy Backtest Completed")
print("=" * 80)

In [ ]:
print("\n" + "=" * 80)
print("Strategy 2: Bollinger Bands Breakout Strategy")
print("=" * 80)

results_bb = {}
metrics_bb = {}

for market_name, df in data_dict.items():
    print(f"\n{'='*60}")
    print(f"Processing Market: {market_name}")
    print('='*60)
    
    is_futures = market_name in ["Soybean Meal", "Corn", "Coke", "Iron Ore"]
    
    macd = macd_series(df["Close"])
    bb = bbands_series(df["Close"])
    feat = pd.concat([df, macd, bb], axis=1)
    
    pos = signals_bbands(feat, long_short=is_futures)
    feat["position"] = pos
    
    result, metrics = backtest(feat, cost_bps=0.0, long_short=is_futures)
    
    results_bb[market_name] = result
    metrics_bb[market_name] = metrics
    
    print(f"\n[Performance Metrics]")
    print(f"  Total Return:      {metrics.total_return:>8.2%}")
    print(f"  CAGR:              {metrics.cagr:>8.2%}")
    print(f"  Sharpe Ratio:      {metrics.sharpe:>8.2f}")
    print(f"  Max Drawdown:      {metrics.maxdd:>8.2%}")
    print(f"  Win Rate:          {metrics.win_rate:>8.2%}")
    print(f"  Number of Trades:  {metrics.n_trades:>8d}")
    
    plot_market_analysis(feat, result, f"{market_name} - Bollinger Bands Strategy")

print("\n" + "=" * 80)
print("✓ Bollinger Bands Strategy Backtest Completed")
print("=" * 80)

In [ ]:
print("\n" + "=" * 80)
print("Strategy 3: MACD + Bollinger Bands Combined Strategy")
print("=" * 80)

results_combo = {}
metrics_combo = {}

for market_name, df in data_dict.items():
    print(f"\n{'='*60}")
    print(f"Processing Market: {market_name}")
    print('='*60)
    
    is_futures = market_name in ["Soybean Meal", "Corn", "Coke", "Iron Ore"]
    
    macd = macd_series(df["Close"])
    bb = bbands_series(df["Close"])
    feat = pd.concat([df, macd, bb], axis=1)
    
    pos = signals_combo(feat, long_short=is_futures)
    feat["position"] = pos
    
    result, metrics = backtest(feat, cost_bps=0.0, long_short=is_futures)
    
    results_combo[market_name] = result
    metrics_combo[market_name] = metrics
    
    print(f"\n[Performance Metrics]")
    print(f"  Total Return:      {metrics.total_return:>8.2%}")
    print(f"  CAGR:              {metrics.cagr:>8.2%}")
    print(f"  Sharpe Ratio:      {metrics.sharpe:>8.2f}")
    print(f"  Max Drawdown:      {metrics.maxdd:>8.2%}")
    print(f"  Win Rate:          {metrics.win_rate:>8.2%}")
    print(f"  Number of Trades:  {metrics.n_trades:>8d}")
    
    plot_market_analysis(feat, result, f"{market_name} - Combined Strategy")

print("\n" + "=" * 80)
print("✓ Combined Strategy Backtest Completed")
print("=" * 80)

In [ ]:
print("\n" + "=" * 80)
print("Performance Summary - All Strategies")
print("=" * 80)

# Create summary data
summary_data = []

for market_name in data_dict.keys():
    for strategy_name, metrics_dict in [
        ("MACD", metrics_macd),
        ("Bollinger Bands", metrics_bb),
        ("Combined", metrics_combo)
    ]:
        if market_name in metrics_dict:
            m = metrics_dict[market_name]
            summary_data.append({
                "Market": market_name,
                "Strategy": strategy_name,
                "Total Return": f"{m.total_return:.2%}",
                "CAGR": f"{m.cagr:.2%}",
                "Sharpe Ratio": f"{m.sharpe:.2f}",
                "Max Drawdown": f"{m.maxdd:.2%}",
                "Win Rate": f"{m.win_rate:.2%}",
                "Trades": m.n_trades
            })

summary_df = pd.DataFrame(summary_data)

# Display by market
print("\n" + "=" * 80)
print("[Performance Comparison by Market]")
print("=" * 80)

for market in data_dict.keys():
    market_summary = summary_df[summary_df["Market"] == market]
    print(f"\n{market}:")
    print(market_summary.to_string(index=False))
    print("-" * 80)

# Display by strategy
print("\n" + "=" * 80)
print("[Performance Comparison by Strategy]")
print("=" * 80)

for strategy in ["MACD", "Bollinger Bands", "Combined"]:
    strategy_summary = summary_df[summary_df["Strategy"] == strategy]
    print(f"\n{strategy}:")
    print(strategy_summary.to_string(index=False))
    print("-" * 80)

# Save to CSV
summary_df.to_csv("outputs/strategy_performance_summary.csv", index=False, encoding="utf-8-sig")
print("\n✓ Performance summary saved to: outputs/strategy_performance_summary.csv")
print("=" * 80)

In [ ]:
print("\n" + "=" * 80)
print("Descriptive Statistics of Strategy Returns")
print("=" * 80)

# Create descriptive statistics for each market and strategy
descriptive_stats = []

for market_name in data_dict.keys():
    for strategy_name, results_dict in [
        ("MACD", results_macd),
        ("Bollinger Bands", results_bb),
        ("Combined", results_combo)
    ]:
        if market_name in results_dict:
            result = results_dict[market_name]
            returns = result['strat_ret'].dropna()
            
            descriptive_stats.append({
                "Market": market_name,
                "Strategy": strategy_name,
                "Mean": returns.mean(),
                "Std Dev": returns.std(),
                "Min": returns.min(),
                "Max": returns.max(),
                "Median": returns.median(),
                "Skewness": returns.skew(),
                "Kurtosis": returns.kurtosis()
            })

desc_df = pd.DataFrame(descriptive_stats)

# Display by market
print("\n" + "=" * 80)
print("[Return Statistics by Market]")
print("=" * 80)

for market in data_dict.keys():
    market_desc = desc_df[desc_df["Market"] == market].copy()
    if len(market_desc) > 0:
        print(f"\n{market}:")
        print("-" * 80)
        for _, row in market_desc.iterrows():
            print(f"  Strategy: {row['Strategy']}")
            print(f"    Mean Return:        {row['Mean']:>10.4%}")
            print(f"    Std Deviation:      {row['Std Dev']:>10.4%}")
            print(f"    Min Return:         {row['Min']:>10.4%}")
            print(f"    Max Return:         {row['Max']:>10.4%}")
            print(f"    Median Return:      {row['Median']:>10.4%}")
            print(f"    Skewness:           {row['Skewness']:>10.4f}")
            print(f"    Kurtosis:           {row['Kurtosis']:>10.4f}")
            print()

# Display by strategy
print("\n" + "=" * 80)
print("[Return Statistics by Strategy]")
print("=" * 80)

for strategy in ["MACD", "Bollinger Bands", "Combined"]:
    strategy_desc = desc_df[desc_df["Strategy"] == strategy].copy()
    if len(strategy_desc) > 0:
        print(f"\n{strategy} Strategy:")
        print("-" * 80)
        print(f"{'Market':<20} {'Mean':<10} {'Std Dev':<10} {'Min':<10} {'Max':<10} {'Median':<10}")
        print("-" * 80)
        for _, row in strategy_desc.iterrows():
            print(f"{row['Market']:<20} {row['Mean']:<10.4%} {row['Std Dev']:<10.4%} "
                  f"{row['Min']:<10.4%} {row['Max']:<10.4%} {row['Median']:<10.4%}")

# Save to CSV
desc_df.to_csv("outputs/return_descriptive_statistics.csv", index=False, encoding="utf-8-sig")
print("\n✓ Descriptive statistics saved to: outputs/return_descriptive_statistics.csv")
print("=" * 80)